In [1]:
import os
import gc 
import neurokit2 as nk
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import psutil
import logging

In [2]:

# Configurar logging para capturar e registrar erros
logging.basicConfig(filename='process_log.log', level=logging.ERROR, 
                    format='%(asctime)s %(levelname)s %(message)s')

def log_memory_usage(stage):
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"[{stage}] Memória usada: {mem_info.rss / 1024 ** 2:.2f} MB")

def extract_and_save_beats_as_images(file_path, output_folder, sampling_rate=128):
    try:
        # Carregar o sinal ECG do arquivo de texto
        with open(file_path, 'r') as file:
            ecg_signal = [float(line.strip()) for line in file]
        
        # Processar o sinal ECG
        ecg_signals, info = nk.ecg_process(ecg_signal, sampling_rate=sampling_rate)
        
        # Extraindo batimentos cardíacos individuais
        r_peaks = info['ECG_R_Peaks']
        
        # Segmentar e salvar batimentos cardíacos individuais como imagens
        base_filename = os.path.splitext(os.path.basename(file_path))[0]
        for i, r_peak in enumerate(r_peaks):
            output_filename = f"{base_filename}-batimento-{i+1}.png"
            output_filepath = os.path.join(output_folder, output_filename)
            
            # Verificar se o arquivo já existe
            if os.path.exists(output_filepath):
                continue
            
            start = r_peak - int(0.2 * sampling_rate)  # 200 ms antes do pico R
            end = r_peak + int(0.4 * sampling_rate)    # 400 ms depois do pico R
            if start >= 0 and end < len(ecg_signal):
                beat_segment = ecg_signal[start:end]
                
                # Plotar o batimento cardíaco e salvar como imagem
                plt.figure(figsize=(6, 3))
                plt.plot(beat_segment)
                plt.title(f"Batimento {i+1} de {base_filename}")
                plt.xlabel("Amostras")
                plt.ylabel("Amplitude")
                plt.savefig(output_filepath)
                plt.close()
                
                # Liberar memória após salvar a imagem
                del beat_segment
                gc.collect()
                
        log_memory_usage(f"Após processamento de {file_path}")
    
    except Exception as e:
        logging.error(f"Erro ao processar o arquivo {file_path}: {e}")

def process_folder(input_folder, output_folder, sampling_rate=128):
    # Criar a pasta de saída se não existir
    os.makedirs(output_folder, exist_ok=True)
    
    # Obter lista de arquivos .txt na pasta de entrada
    files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]
    
    # Processar cada arquivo na pasta de entrada com barra de progresso
    for filename in tqdm(files, desc="Processando arquivos"):
        file_path = os.path.join(input_folder, filename)
        extract_and_save_beats_as_images(file_path, output_folder, sampling_rate)

# Defina as pastas de entrada e saída
input_folder = "/home/leo-vitor/Documentos/UFC/biosinais/Exames_24h_recortados/3.0"
output_folder = "/home/leo-vitor/Documentos/UFC/biosinais/Exames_24h_recortados/3.0/output"

# Processar a pasta de entrada
process_folder(input_folder, output_folder)

Processando arquivos:   0%|          | 0/37 [00:00<?, ?it/s]/home/leo-vitor/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_fixpeaks.py:307: RuntimeWarning: divide by zero encountered in divide
  mrrs /= th2
Processando arquivos:   3%|▎         | 1/37 [02:02<1:13:39, 122.76s/it]

[Após processamento de /home/leo-vitor/Documentos/UFC/biosinais/Exames_24h_recortados/3.0/090.txt] Memória usada: 1612.71 MB
